In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.datasets import imdb
from tensorflow.keras.utils import pad_sequences

In [ ]:
# ----------------------------- Config ---------------------------------
VOCAB_SIZE = 10000   # keep the 10,000 most frequent words
MAX_LEN = 200        # pad / truncate every review to 200 tokens
EMBED_DIM = 128
EPOCHS = 5
BATCH_SIZE = 64
SEED = 42

np.random.seed(SEED)

In [ ]:
# ----------------------------- Data -----------------------------------
print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)
print(f"Train reviews: {len(x_train)} | Test reviews: {len(x_test)}")

x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding="post", truncating="post")
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding="post", truncating="post")
print("Padded shapes:", x_train.shape, x_test.shape)

In [ ]:
# ----------------------------- Models ---------------------------------
def build_model(kind):
    """kind: 'rnn' (SimpleRNN) or 'lstm' (Bidirectional LSTM)."""
    model = models.Sequential(name=f"imdb_{kind}")
    model.add(layers.Input(shape=(MAX_LEN,)))
    model.add(layers.Embedding(VOCAB_SIZE, EMBED_DIM))
    if kind == "rnn":
        model.add(layers.SimpleRNN(64))
    else:
        model.add(layers.Bidirectional(layers.LSTM(64)))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(32, activation="relu"))
    model.add(layers.Dense(1, activation="sigmoid"))  # probability of positive
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


def train_and_evaluate(kind):
    print(f"\n{'=' * 60}\nTraining {kind.upper()}\n{'=' * 60}")
    model = build_model(kind)
    model.summary()

    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                           restore_best_weights=True)],
        verbose=1,
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"\n{kind.upper()} -> Test accuracy: {acc:.4f} | Test loss: {loss:.4f}")

    y_pred = (model.predict(x_test, verbose=0) >= 0.5).astype(int).ravel()
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(cmap="Blues")
    plt.title(f"{kind.upper()} confusion matrix")
    plt.savefig(f"confusion_{kind}.png", dpi=120)
    plt.show()

    model.save(f"imdb_{kind}.keras")
    return model, history, acc


results = {}
for kind in ["rnn", "lstm"]:
    results[kind] = train_and_evaluate(kind)

In [ ]:
# ------------------------ Training curves -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for kind, (_, hist, _) in results.items():
    axes[0].plot(hist.history["val_accuracy"], label=f"{kind.upper()} val")
    axes[0].plot(hist.history["accuracy"], "--", label=f"{kind.upper()} train")
    axes[1].plot(hist.history["val_loss"], label=f"{kind.upper()} val")
    axes[1].plot(hist.history["loss"], "--", label=f"{kind.upper()} train")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()

print("\nTest accuracy comparison:")
for kind, (_, _, acc) in results.items():
    print(f"  {kind.upper():5s}: {acc:.4f}")

In [ ]:
# ------------------- Predict on your own reviews ----------------------
word_index = imdb.get_word_index()


def encode_review(text):
    """Convert raw text into the same integer format Keras uses for IMDB."""
    ids = [1]  # 1 = start token
    for word in text.lower().replace("<br />", " ").split():
        word = "".join(c for c in word if c.isalnum() or c == "'")
        idx = word_index.get(word)
        # Keras shifts real word indices by 3 (0=pad, 1=start, 2=unknown)
        if idx is not None and idx + 3 < VOCAB_SIZE:
            ids.append(idx + 3)
        else:
            ids.append(2)
    return pad_sequences([ids], maxlen=MAX_LEN, padding="post", truncating="post")


def predict_sentiment(text, kind="lstm"):
    model = results[kind][0]
    p = float(model.predict(encode_review(text), verbose=0)[0][0])
    return ("Positive" if p >= 0.5 else "Negative"), p


samples = [
    "This movie was absolutely fantastic, the acting and story were brilliant!",
    "Terrible film. Boring plot, awful acting, a complete waste of time.",
    "It was okay, a few good moments but overall quite forgettable.",
]
print("\nSample predictions (LSTM):")
for s in samples:
    label, p = predict_sentiment(s)
    print(f"  [{label}  P(positive)={p:.2f}] {s}")